# Published baseline with its own LM (Colab)

`anuragshas/wav2vec2-xls-r-300m-ur-cv9-with-lm` ships a KenLM language model and its own `alpha`/`beta`. Decoding it needs `kenlm`, which does not install on Windows, so this one row runs here.

Vanilla: nothing is trained or tuned; the model's LM and settings are used as published, with pyctcdecode's default beam width (100). The greedy row for the same checkpoint was run on the RTX 4060.

Use a GPU runtime (Runtime → Change runtime type → T4 GPU), then Runtime → Run all.

In [ ]:
import os, subprocess
# KenLM's CMakeLists asks for an old CMake version; CMake 4 refuses that unless told the minimum policy
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
%cd /content
![ -d urdu-asr-kathbath ] || git clone -q --depth 1 https://github.com/r4fo/urdu-asr-kathbath.git
%cd /content/urdu-asr-kathbath
!git pull -q
# Colab runs Python 3.13, where kenlm and pyctcdecode do not build. So the LM steps run in a Python 3.11
# environment with the same pinned packages as requirements.txt (Python 3.11 locally too), plus the LM decoder.
!apt-get -qq update > /dev/null; apt-get -qq install -y build-essential cmake libboost-program-options-dev libboost-system-dev libboost-thread-dev libboost-test-dev libeigen3-dev zlib1g-dev libbz2-dev liblzma-dev > /dev/null
!pip install -q uv
![ -x /content/py311/bin/python ] || uv venv -q -p 3.11 /content/py311
!uv pip install -q -p /content/py311/bin/python -r requirements.txt pyctcdecode==0.5.0 kenlm==0.2.0
PY = "/content/py311/bin/python"
# stops "Run all" here if anything is missing
subprocess.run([PY, "-c", "import torch, kenlm, pyctcdecode; assert torch.cuda.is_available(), 'no GPU'; print('kenlm and pyctcdecode OK, GPU:', torch.cuda.get_device_name(0))"], check=True)

In [ ]:
# Test, once (no tuning: the model's own LM and alpha/beta)
!{PY} models/wav2vec2_finetune/transcribe.py --model anuragshas/wav2vec2-xls-r-300m-ur-cv9-with-lm --model-lm --beam-width 100 --manifest data/splits/test.csv --out results/predictions/anuragshas_w2v2_ur_lm.csv --system anuragshas_w2v2_ur_lm

In [ ]:
!{PY} src/evaluate.py --split data/splits/test.csv --pred results/predictions/anuragshas_w2v2_ur_lm.csv

In [ ]:
# Download and commit from your own machine
!zip -q anuragshas_lm_outputs.zip results/predictions/anuragshas_w2v2_ur_lm.csv results/timing/anuragshas_w2v2_ur_lm.csv
from google.colab import files
files.download("anuragshas_lm_outputs.zip")